In [1]:
# Ovella AI System - Prototype for AI Adaptive Learning Model
#from google.colab import files #later add this as a new cell for using real medical data (expecting accuracy to go down a bit since its synthetic => REAL with different parameters and without direct obvious correlations)


import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd

# For screening
from sklearn.ensemble import RandomForestClassifier

# For ML
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# For Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

print("Ovella: Current MVP for Adaptive Learning Model \n")

# PART 1: EARLY SCREENING MODEL (Synthetic Controlled Data)
# --------------------------------------------------------

n_samples = 2000

diagnosed = np.random.binomial(1, 0.5, n_samples)

def generate_symptom(prob_diag, prob_nondiag):
    return np.where(
        diagnosed == 1,
        np.random.binomial(1, prob_diag, n_samples),
        np.random.binomial(1, prob_nondiag, n_samples)
    )

data = pd.DataFrame({
    "diagnosed": diagnosed,
    "dysmenorrhea": generate_symptom(0.80, 0.10),
    "cramping": generate_symptom(0.85, 0.15),
    "painful_period_cramps": generate_symptom(0.75, 0.10),
    "fatigue": generate_symptom(0.70, 0.20),
    "heavy_bleeding": generate_symptom(0.78, 0.25)
})

X = data.drop("diagnosed", axis=1)
y = data["diagnosed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model_screen = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model_screen.fit(X_train, y_train)

print("Screening Model Performance:")
print(classification_report(y_test, model_screen.predict(X_test)))


# PART 2: MULTIMODAL DEEP LEARNING TENS OPTIMIZATION MODEL
# --------------------------------------------------------

n_samples = 3000
signal_length = 100

# Objective physiological signal (simulated waveform)
X_objective = np.random.normal(0, 1, (n_samples, signal_length))

# Contextual data (aligned with screening logic)
cycle_phase = np.random.choice([0,1,2,3], n_samples)
stress = np.random.beta(2,2, n_samples)
prior_relief = np.random.beta(2,5, n_samples)
fatigue = np.random.beta(2,3, n_samples)
period_day = np.random.randint(1, 29, n_samples)

pain_level = np.clip(np.random.normal(6, 2, n_samples), 0, 10) / 10

X_context = np.column_stack([
    pain_level,
    cycle_phase/3,
    stress,
    prior_relief,
    fatigue,
    period_day/28
])

# Synthetic optimal TENS outputs (bounded real-world values)

amplitude_mA = 10 + (40 * pain_level) + (5 * stress)
frequency_Hz = np.where(
    fatigue > 0.6,
    2 + 8 * pain_level,
    60 + 60 * pain_level
)
pulse_width_us = 100 + (250 * pain_level)

amplitude_mA = np.clip(amplitude_mA, 10, 60)
frequency_Hz = np.clip(frequency_Hz, 2, 120)
pulse_width_us = np.clip(pulse_width_us, 100, 400)

y_tens = np.column_stack([
    amplitude_mA,
    frequency_Hz,
    pulse_width_us
])

X_obj_train, X_obj_test, X_ctx_train, X_ctx_test, y_train_tens, y_test_tens = train_test_split(
    X_objective, X_context, y_tens,
    test_size=0.2,
    random_state=42
)

# Multimodal architecture

input_obj = Input(shape=(signal_length,))
x1 = Dense(128, activation='relu')(input_obj)
x1 = Dense(64, activation='relu')(x1)

input_ctx = Input(shape=(X_context.shape[1],))
x2 = Dense(32, activation='relu')(input_ctx)
x2 = Dense(16, activation='relu')(x2)

merged = Concatenate()([x1, x2])
merged = Dense(64, activation='relu')(merged)
merged = Dropout(0.2)(merged)
output = Dense(3, activation='linear')(merged)

model_tens = Model(inputs=[input_obj, input_ctx], outputs=output)
model_tens.compile(optimizer=Adam(0.001), loss='mse')

model_tens.fit(
    [X_obj_train, X_ctx_train],
    y_train_tens,
    epochs=15,
    batch_size=32,
    verbose=0
)

print("TENS Multimodal Model Trained.")


# PART 3: INFERENCE (SCREENING + TENS OUTPUT)
# --------------------------------------------------------

# Example new patient input using synthetic features
new_symptoms = pd.DataFrame([{
    "dysmenorrhea": 1,
    "cramping": 1,
    "painful_period_cramps": 1,
    "fatigue": 1,
    "heavy_bleeding": 1,

    # Synthetic features for interaction terms
    "Chronic_Pain_Level": np.random.rand(),
    "Menstrual_Irregularity": np.random.rand(),
    "Hormone_Level_Abnormality": np.random.rand()
}])

# Compute interaction terms safely
new_symptoms["Pain_Irregularity_Interaction"] = (
    new_symptoms["Chronic_Pain_Level"] *
    new_symptoms["Menstrual_Irregularity"]
)

new_symptoms["Hormone_Pain_Interaction"] = (
    new_symptoms["Hormone_Level_Abnormality"] *
    new_symptoms["Chronic_Pain_Level"]
)

# Keep only the columns used by the screening model for prediction
screen_input = new_symptoms[["dysmenorrhea", "cramping", "painful_period_cramps", "fatigue", "heavy_bleeding"]]

# Run screening
screen_probability = model_screen.predict_proba(screen_input)[0][1]
screen_prediction = model_screen.predict(screen_input)[0]

if screen_prediction == 1:
    screening_result = (
        f"High likelihood ({screen_probability*100:.1f}% risk). "
        "(AI-based early screening result — not a medical diagnosis.) "
        "Recommend specialist evaluation."
    )
else:
    screening_result = (
        f"Lower likelihood ({screen_probability*100:.1f}% risk). "
        "Continue monitoring symptoms."
    )

print("\nScreening Result:")
print(screening_result)

# Generate personalized TENS settings using the same synthetic approach
new_objective = np.random.normal(0, 1, (1, signal_length))
new_context = np.array([[
    new_symptoms["Chronic_Pain_Level"].values[0],  # pain level
    np.random.rand(),                              # cycle_phase
    np.random.rand(),                              # stress
    np.random.rand(),                              # prior_relief
    new_symptoms["fatigue"].values[0],            # fatigue
    np.random.rand()                               # period_day
]])

new_objective = tf.convert_to_tensor(new_objective, dtype=tf.float32)
new_context = tf.convert_to_tensor(new_context, dtype=tf.float32)

predicted_tens = model_tens([new_objective, new_context], training=False).numpy()[0]

print("\nRecommended TENS Settings:")
print("Note: Parameters are generated within illustrative prototype bounds and are not medical recommendations.")
print(f"Amplitude: {predicted_tens[0]:.2f} mA")
print(f"Frequency: {predicted_tens[1]:.2f} Hz")
print(f"Pulse Width: {predicted_tens[2]:.2f} µs")


# PART 4: ADAPTIVE CLOSED-LOOP OPTIMIZATION
# --------------------------------------------------------

print("\nAdaptive Closed-Loop Optimization\n")

def simulate_pain_response(pain, amplitude, frequency, pulse_width):

    effect = (
        0.2 * (amplitude / 60) +
        0.2 * (frequency / 120) +
        0.2 * (pulse_width / 400)
    )

    noise = np.random.normal(0, 0.03)
    pain_after = pain - effect + noise
    pain_after = np.clip(pain_after, 0, 1)

    return pain_after


current_pain = float(new_context.numpy()[0][0])
stress = float(new_context.numpy()[0][2])
fatigue = float(new_context.numpy()[0][4])

amplitude = predicted_tens[0]
frequency = predicted_tens[1]
pulse_width = predicted_tens[2]

sessions = 5
learning_rate = 0.6

for session in range(sessions):

    pain_before = current_pain

    pain_after = simulate_pain_response(
        pain_before,
        amplitude,
        frequency,
        pulse_width
    )

    reward = pain_before - pain_after

    amplitude += learning_rate * 5 * reward
    frequency += learning_rate * (10 * reward if fatigue < 0.6 else -5 * reward)
    pulse_width += learning_rate * 20 * reward

    amplitude = np.clip(amplitude, 10, 60)
    frequency = np.clip(frequency, 2, 120)
    pulse_width = np.clip(pulse_width, 100, 400)

    current_pain = pain_after

    print(f"Session {session+1}")
    print(f"  Pain: {pain_before:.2f} → {pain_after:.2f}")
    print(f"  Updated TENS Settings:")
    print(f"    Amplitude: {amplitude:.2f} mA")
    print(f"    Frequency: {frequency:.2f} Hz")
    print(f"    Pulse Width: {pulse_width:.2f} µs\n_")

print("\nOvella Adaptive Session Complete.")


Ovella: Current MVP for Adaptive Learning Model 

Screening Model Performance:
              precision    recall  f1-score   support

           0       0.91      0.96      0.94       197
           1       0.96      0.91      0.94       203

    accuracy                           0.94       400
   macro avg       0.94      0.94      0.94       400
weighted avg       0.94      0.94      0.94       400

TENS Multimodal Model Trained.

Screening Result:
High likelihood (99.5% risk). (AI-based early screening result — not a medical diagnosis.) Recommend specialist evaluation.

Recommended TENS Settings:
Note: Parameters are generated within illustrative prototype bounds and are not medical recommendations.
Amplitude: 37.26 mA
Frequency: 85.40 Hz
Pulse Width: 259.40 µs

Adaptive Closed-Loop Optimization

Session 1
  Pain: 0.61 → 0.24
  Updated TENS Settings:
    Amplitude: 38.35 mA
    Frequency: 84.30 Hz
    Pulse Width: 263.78 µs
_
Session 2
  Pain: 0.24 → 0.00
  Updated TENS Settings:
 